In [2]:
import os
import time
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, pmap, grad, random
from jax.experimental import pjit
from jax.sharding import Mesh, PositionalSharding
from functools import partial
import yfinance as yf

# ========================================
# Live Data Fetching (Modified)
# ========================================
ticker_symbol = "BTC-USD"
ticker = yf.Ticker(ticker_symbol)
# Fetch the last 5 days of data at 5-minute intervals.
live_data = ticker.history(period="5d", interval="5m")
print("Live data preview:")
print(live_data.head())

# Save to CSV for inspection.
live_data.to_csv("live_btc_data.csv")
print("Saved live data to live_btc_data.csv")

# ========================================
# Trading Data Ingestion & Preprocessing
# ========================================
def load_live_data(dataframe, column="Close"):
    data = dataframe[column].values.astype(np.float32)
    if data.size == 0:
        raise ValueError("No data found in the specified column. Check your ticker, period, or interval.")
    # Normalize using z-score normalization.
    data = (data - np.mean(data)) / np.std(data)
    return data

raw_live_data_np = load_live_data(live_data, column="Close")
raw_live_data = jnp.array(raw_live_data_np)

# ========================================
# Pipeline Configuration (Small Scale for Trading)
# ========================================
PIPE_MAX_RECURSION_DEPTH    = 100_000      # Maximum recursion depth for pipeline
PIPE_TOTAL_DEPTH            = 100_000      # Total recursion depth (full fusion)
PIPE_OPTIMAL_DEPTH_STEP     = PIPE_TOTAL_DEPTH  # Full fusion in one call
PIPE_DIMENSIONAL_CONSTRAINT = 0.8
# Use up to all available live data.
MAX_PIPE_BATCH_SIZE         = raw_live_data.shape[0]

PIPE_VAL_CLAMP_LOW  = -100.0
PIPE_VAL_CLAMP_HIGH =  100.0

# Use up to 8 devices (or available devices)
PIPE_NUM_DEVICES = min(8, jax.device_count())
devices = jax.devices()[:PIPE_NUM_DEVICES]
mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

DATA_LAKE_DIR = "datalake"
if not os.path.exists(DATA_LAKE_DIR):
    os.makedirs(DATA_LAKE_DIR)

# -------------------------------
# Pipeline Functions
# -------------------------------
@jit
def pipe_dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def pipe_dppu_with_dynamic_pi_phi(x, depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = pipe_stabilize_depth(jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = pipe_dynamic_pi(i, scale_factor)
        phi_dyn = pipe_dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, PIPE_VAL_CLAMP_LOW, PIPE_VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

def pipe_branch_recycle(x, num_branches=2, branch_depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    xs = jnp.stack([x] * num_branches, axis=0)
    branch_fn = vmap(lambda xi: pipe_dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
    branch_outputs = branch_fn(xs)
    return jnp.sum(branch_outputs, axis=0)

@partial(pjit.pjit,
         in_shardings=(sharding,),
         out_shardings=sharding,
         static_argnames=("num_branches", "branch_depth", "scale_factor"))
def pipe_process_full(x, num_branches, branch_depth, scale_factor):
    return pipe_branch_recycle(x, num_branches=num_branches, branch_depth=branch_depth, scale_factor=scale_factor)

def run_pipeline(raw_data):
    # Limit to MAX_PIPE_BATCH_SIZE samples.
    data = raw_data[:MAX_PIPE_BATCH_SIZE]
    sharded_input = jax.device_put(data, sharding)
    start_time = time.time()
    final_output = pipe_process_full(sharded_input, 2, PIPE_OPTIMAL_DEPTH_STEP, 1.0)
    final_output = jax.device_get(final_output)
    jax.block_until_ready(final_output)
    elapsed = time.time() - start_time
    mean_val = float(jnp.mean(final_output))
    return final_output, elapsed, mean_val

def save_to_data_lake(data, filename="processed_trading_data.npy"):
    filepath = os.path.join(DATA_LAKE_DIR, filename)
    np.save(filepath, np.array(data))
    print(f"Saved processed data to {filepath}")

# ========================================
# RNN Configuration & Functions (for Trading)
# ========================================
input_dim = 1
hidden_dim = 32
output_dim = 1
seq_length = 100  # Each sequence will be 100 samples

def get_num_sequences(data_length, seq_length):
    return data_length // seq_length

def init_rnn_params(key, input_dim, hidden_dim, output_dim):
    k1, k2, k3, k4 = random.split(key, 4)
    W_xh = random.normal(k1, (input_dim, hidden_dim)) * 0.1
    W_hh = random.normal(k2, (hidden_dim, hidden_dim)) * 0.1
    b_h = jnp.zeros((hidden_dim,))
    W_hy = random.normal(k3, (hidden_dim, output_dim)) * 0.1
    b_y = jnp.zeros((output_dim,))
    return {"W_xh": W_xh, "W_hh": W_hh, "b_h": b_h, "W_hy": W_hy, "b_y": b_y}

def rnn_step(params, h, x):
    h_next = jnp.tanh(jnp.dot(x, params["W_xh"]) + jnp.dot(h, params["W_hh"]) + params["b_h"])
    return h_next

def rnn_forward(params, inputs):
    # inputs: (seq_length, input_dim)
    def step_fn(h, x):
        h_new = rnn_step(params, h, x)
        return h_new, h_new
    h0 = jnp.zeros((hidden_dim,))
    final_h, _ = lax.scan(step_fn, h0, inputs)
    output = jnp.dot(final_h, params["W_hy"]) + params["b_y"]
    return output

def rnn_loss_fn(params, batch_inputs, batch_targets):
    def single_loss(inputs, target):
        pred = rnn_forward(params, inputs)
        return jnp.mean((pred - target) ** 2)
    losses = jax.vmap(single_loss)(batch_inputs, batch_targets)
    return jnp.mean(losses)

@jit
def rnn_train_step(params, batch_inputs, batch_targets, learning_rate=0.001):
    grads = grad(rnn_loss_fn)(params, batch_inputs, batch_targets)
    new_params = {k: params[k] - learning_rate * grads[k] for k in params}
    loss = rnn_loss_fn(params, batch_inputs, batch_targets)
    return new_params, loss

# ========================================
# Main: End-to-End Live Trading DRNN System
# ========================================
print("Running processing pipeline on live trading data...")
processed_live_data, pipe_elapsed, pipe_mean = run_pipeline(raw_live_data)
print(f"Pipeline completed in {pipe_elapsed:.2f} sec with mean output {pipe_mean:.6f}")
save_to_data_lake(processed_live_data, filename="processed_trading_data.npy")

# Prepare data for RNN:
total_samples = processed_live_data.shape[0]
num_sequences = get_num_sequences(total_samples, seq_length)
if num_sequences == 0:
    print("Not enough samples to form a sequence. Adjust seq_length or wait for more data.")
else:
    rnn_live_inputs = processed_live_data[:num_sequences * seq_length].reshape((num_sequences, seq_length, 1))
    # For next-value prediction, use the last value in each sequence as the target.
    rnn_live_targets = rnn_live_inputs[:, -1, :]
    print(f"RNN live training data: {num_sequences} sequences of length {seq_length}")

    # For demonstration, run inference on a sample sequence.
    sample_input = rnn_live_inputs[0]
    sample_prediction = rnn_forward(rnn_params, sample_input)
    print("Live sample RNN prediction:", sample_prediction)
    print("Live sample RNN target:", rnn_live_targets[0])



Live data preview:
                                   Open          High           Low  \
Datetime                                                              
2025-02-21 00:00:00+00:00  98338.843750  98496.976562  98338.843750   
2025-02-21 00:05:00+00:00  98441.820312  98498.421875  98441.820312   
2025-02-21 00:10:00+00:00  98498.492188  98513.187500  98490.437500   
2025-02-21 00:15:00+00:00  98468.687500  98468.687500  98451.382812   
2025-02-21 00:20:00+00:00  98411.937500  98411.937500  98367.687500   

                                  Close     Volume  Dividends  Stock Splits  
Datetime                                                                     
2025-02-21 00:00:00+00:00  98486.226562          0        0.0           0.0  
2025-02-21 00:05:00+00:00  98498.421875   25540608        0.0           0.0  
2025-02-21 00:10:00+00:00  98490.437500          0        0.0           0.0  
2025-02-21 00:15:00+00:00  98451.382812  787910656        0.0           0.0  
2025-02-21 00:2

In [4]:
import os
import time
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, grad, random
from jax.experimental import pjit
from jax.sharding import Mesh, PositionalSharding
from functools import partial
import yfinance as yf

# ========================================
# Live Data Fetching: Increase Sample Size
# ========================================
ticker_symbol = "BTC-USD"
ticker = yf.Ticker(ticker_symbol)
# Fetch 60 days of intraday data at 5-minute intervals.
live_data = ticker.history(period="60d", interval="5m")
print("Live data preview:")
print(live_data.head())

# Save the live data to CSV for inspection.
live_data.to_csv("live_btc_data.csv")
print("Saved live data to live_btc_data.csv")

# ========================================
# Trading Data Ingestion & Preprocessing
# ========================================
def load_live_data(dataframe, column="Close"):
    data = dataframe[column].values.astype(np.float32)
    if data.size == 0:
        raise ValueError("No data found in the specified column. Check your ticker, period, or interval.")
    # Normalize using z-score normalization.
    data = (data - np.mean(data)) / np.std(data)
    return data

raw_live_data_np = load_live_data(live_data, column="Close")
raw_live_data = jnp.array(raw_live_data_np)
print(f"Loaded live data of length: {raw_live_data.shape[0]}")

# ========================================
# Pipeline Configuration (Small Scale for Trading)
# ========================================
PIPE_MAX_RECURSION_DEPTH    = 100_000      # Maximum recursion depth for pipeline
PIPE_TOTAL_DEPTH            = 100_000      # Total recursion depth (full fusion)
PIPE_OPTIMAL_DEPTH_STEP     = PIPE_TOTAL_DEPTH  # Full fusion in one call
PIPE_DIMENSIONAL_CONSTRAINT = 0.8
# Use all available live data samples.
MAX_PIPE_BATCH_SIZE         = raw_live_data.shape[0]

PIPE_VAL_CLAMP_LOW  = -100.0
PIPE_VAL_CLAMP_HIGH =  100.0

# Use up to 8 devices (or available devices)
PIPE_NUM_DEVICES = min(8, jax.device_count())
devices = jax.devices()[:PIPE_NUM_DEVICES]
mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

DATA_LAKE_DIR = "datalake"
if not os.path.exists(DATA_LAKE_DIR):
    os.makedirs(DATA_LAKE_DIR)

# -------------------------------
# Pipeline Functions
# -------------------------------
@jit
def pipe_dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def pipe_dppu_with_dynamic_pi_phi(x, depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = pipe_stabilize_depth(jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = pipe_dynamic_pi(i, scale_factor)
        phi_dyn = pipe_dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, PIPE_VAL_CLAMP_LOW, PIPE_VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

def pipe_branch_recycle(x, num_branches=2, branch_depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    xs = jnp.stack([x] * num_branches, axis=0)
    branch_fn = vmap(lambda xi: pipe_dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
    branch_outputs = branch_fn(xs)
    return jnp.sum(branch_outputs, axis=0)

# Use pjit with positional arguments for static parameters.
@partial(pjit.pjit,
         in_shardings=(sharding,),
         out_shardings=sharding,
         static_argnames=("num_branches", "branch_depth", "scale_factor"))
def pipe_process_full(x, num_branches, branch_depth, scale_factor):
    return pipe_branch_recycle(x, num_branches=num_branches, branch_depth=branch_depth, scale_factor=scale_factor)

def run_pipeline(raw_data):
    # Limit to MAX_PIPE_BATCH_SIZE samples.
    data = raw_data[:MAX_PIPE_BATCH_SIZE]
    sharded_input = jax.device_put(data, sharding)
    start_time = time.time()
    final_output = pipe_process_full(sharded_input, 2, PIPE_OPTIMAL_DEPTH_STEP, 1.0)
    final_output = jax.device_get(final_output)
    jax.block_until_ready(final_output)
    elapsed = time.time() - start_time
    mean_val = float(jnp.mean(final_output))
    return final_output, elapsed, mean_val

def save_to_data_lake(data, filename="processed_trading_data.npy"):
    filepath = os.path.join(DATA_LAKE_DIR, filename)
    np.save(filepath, np.array(data))
    print(f"Saved processed data to {filepath}")

# ========================================
# RNN Configuration & Functions (for Trading)
# ========================================
input_dim = 1
hidden_dim = 32
output_dim = 1
seq_length = 100  # Each sequence will be 100 samples

def get_num_sequences(data_length, seq_length):
    return data_length // seq_length

def init_rnn_params(key, input_dim, hidden_dim, output_dim):
    k1, k2, k3, k4 = random.split(key, 4)
    W_xh = random.normal(k1, (input_dim, hidden_dim)) * 0.1
    W_hh = random.normal(k2, (hidden_dim, hidden_dim)) * 0.1
    b_h = jnp.zeros((hidden_dim,))
    W_hy = random.normal(k3, (hidden_dim, output_dim)) * 0.1
    b_y = jnp.zeros((output_dim,))
    return {"W_xh": W_xh, "W_hh": W_hh, "b_h": b_h, "W_hy": W_hy, "b_y": b_y}

def rnn_step(params, h, x):
    h_next = jnp.tanh(jnp.dot(x, params["W_xh"]) + jnp.dot(h, params["W_hh"]) + params["b_h"])
    return h_next

def rnn_forward(params, inputs):
    # inputs: (seq_length, input_dim)
    def step_fn(h, x):
        h_new = rnn_step(params, h, x)
        return h_new, h_new
    h0 = jnp.zeros((hidden_dim,))
    final_h, _ = lax.scan(step_fn, h0, inputs)
    output = jnp.dot(final_h, params["W_hy"]) + params["b_y"]
    return output

def rnn_loss_fn(params, batch_inputs, batch_targets):
    def single_loss(inputs, target):
        pred = rnn_forward(params, inputs)
        return jnp.mean((pred - target) ** 2)
    losses = jax.vmap(single_loss)(batch_inputs, batch_targets)
    return jnp.mean(losses)

@jit
def rnn_train_step(params, batch_inputs, batch_targets, learning_rate=0.001):
    grads = grad(rnn_loss_fn)(params, batch_inputs, batch_targets)
    new_params = {k: params[k] - learning_rate * grads[k] for k in params}
    loss = rnn_loss_fn(params, batch_inputs, batch_targets)
    return new_params, loss

# For demonstration, initialize the RNN randomly.
key = random.PRNGKey(0)
rnn_params = init_rnn_params(key, input_dim, hidden_dim, output_dim)

# ========================================
# Main: End-to-End Live Trading DRNN System
# ========================================
print("Running processing pipeline on live trading data...")
processed_live_data, pipe_elapsed, pipe_mean = run_pipeline(raw_live_data)
print(f"Pipeline completed in {pipe_elapsed:.2f} sec with mean output {pipe_mean:.6f}")
save_to_data_lake(processed_live_data, filename="processed_trading_data.npy")

# Prepare data for RNN:
total_samples = processed_live_data.shape[0]
num_sequences = get_num_sequences(total_samples, seq_length)
if num_sequences == 0:
    print("Not enough samples to form a sequence. Adjust seq_length or wait for more data.")
else:
    rnn_live_inputs = processed_live_data[:num_sequences * seq_length].reshape((num_sequences, seq_length, 1))
    # For next-value prediction, use the last value in each sequence as the target.
    rnn_live_targets = rnn_live_inputs[:, -1, :]
    print(f"RNN live training data: {num_sequences} sequences of length {seq_length}")

    # For demonstration, run inference on a sample sequence.
    sample_input = rnn_live_inputs[0]
    sample_prediction = rnn_forward(rnn_params, sample_input)
    print("Live sample RNN prediction:", sample_prediction)
    print("Live sample RNN target:", rnn_live_targets[0])



Live data preview:
                                   Open          High           Low  \
Datetime                                                              
2024-12-28 00:00:00+00:00  94159.828125  94243.117188  94159.828125   
2024-12-28 00:05:00+00:00  94236.851562  94289.882812  94236.851562   
2024-12-28 00:10:00+00:00  94230.492188  94253.515625  94183.570312   
2024-12-28 00:15:00+00:00  94260.320312  94290.648438  94260.320312   
2024-12-28 00:20:00+00:00  94331.656250  94331.656250  94280.679688   

                                  Close    Volume  Dividends  Stock Splits  
Datetime                                                                    
2024-12-28 00:00:00+00:00  94219.835938         0        0.0           0.0  
2024-12-28 00:05:00+00:00  94239.468750  32845824        0.0           0.0  
2024-12-28 00:10:00+00:00  94253.515625         0        0.0           0.0  
2024-12-28 00:15:00+00:00  94278.257812         0        0.0           0.0  
2024-12-28 00:20:00+0

In [ ]:
import os
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap
from jax.experimental import pjit
from jax.sharding import Mesh, PositionalSharding
from functools import partial

# For synthetic testing, set a large dataset size.
SYNTHETIC_DATA_SIZE = 10_000_000  # 10 million samples

# Create a synthetic dataset (simulate normalized data).
# For example, using random normal data.
synthetic_data_np = np.random.randn(SYNTHETIC_DATA_SIZE).astype(np.float32)
synthetic_data = jnp.array(synthetic_data_np)
print(f"Generated synthetic data of length: {synthetic_data.shape[0]}")

# ========================================
# Pipeline Configuration (Stress Test)
# ========================================
PIPE_MAX_RECURSION_DEPTH    = 100_000      # Maximum recursion depth for pipeline
PIPE_TOTAL_DEPTH            = 100_000      # Total recursion depth (full fusion)
PIPE_OPTIMAL_DEPTH_STEP     = PIPE_TOTAL_DEPTH  # Full fusion in one call
PIPE_DIMENSIONAL_CONSTRAINT = 0.8
# Use the entire synthetic dataset.
MAX_PIPE_BATCH_SIZE         = synthetic_data.shape[0]

PIPE_VAL_CLAMP_LOW  = -100.0
PIPE_VAL_CLAMP_HIGH =  100.0

# Use up to 8 devices (or available devices)
PIPE_NUM_DEVICES = min(8, jax.device_count())
devices = jax.devices()[:PIPE_NUM_DEVICES]
mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

# -------------------------------
# Pipeline Functions (Same as Before)
# -------------------------------
@jit
def pipe_dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def pipe_dppu_with_dynamic_pi_phi(x, depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = pipe_stabilize_depth(jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = pipe_dynamic_pi(i, scale_factor)
        phi_dyn = pipe_dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, PIPE_VAL_CLAMP_LOW, PIPE_VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

def pipe_branch_recycle(x, num_branches=2, branch_depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    xs = jnp.stack([x] * num_branches, axis=0)
    branch_fn = vmap(lambda xi: pipe_dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
    branch_outputs = branch_fn(xs)
    return jnp.sum(branch_outputs, axis=0)

@partial(pjit.pjit,
         in_shardings=(sharding,),
         out_shardings=sharding,
         static_argnames=("num_branches", "branch_depth", "scale_factor"))
def pipe_process_full(x, num_branches, branch_depth, scale_factor):
    return pipe_branch_recycle(x, num_branches=num_branches, branch_depth=branch_depth, scale_factor=scale_factor)

def run_pipeline(raw_data):
    data = raw_data[:MAX_PIPE_BATCH_SIZE]
    sharded_input = jax.device_put(data, sharding)
    start_time = time.time()
    final_output = pipe_process_full(sharded_input, 2, PIPE_OPTIMAL_DEPTH_STEP, 1.0)
    final_output = jax.device_get(final_output)
    jax.block_until_ready(final_output)
    elapsed = time.time() - start_time
    mean_val = float(jnp.mean(final_output))
    return final_output, elapsed, mean_val

# ========================================
# Threshold Test: Run the Pipeline on Synthetic Data
# ========================================
processed_synthetic_data, pipeline_time, pipeline_mean = run_pipeline(synthetic_data)
print(f"Processed {MAX_PIPE_BATCH_SIZE} samples in {pipeline_time:.2f} sec with mean output {pipeline_mean:.6f}")
print(f"Throughput: {MAX_PIPE_BATCH_SIZE / pipeline_time:.2f} samples per second")


Generated synthetic data of length: 10000000
